# DataMate · Agent Locked-In1 (Zalo AI Challenge 2026, Track 2)

**Bài toán:** trả lời câu hỏi tiếng Việt về dữ liệu kinh doanh của một cửa hàng (các bảng csv / xlsx / sqlite và một README). Đáp án có kiểu
cố định (số, phần trăm, chuỗi, danh sách, ngày, bảng…), hoặc bỏ trống khi không chắc.

**Cách xử lý:**
- Mô hình ngôn ngữ (Qwen3.5-9B + LoRA) **viết SQL**, còn DuckDB **thực hiện phép tính** trên dữ liệu thật.
- Mỗi câu hỏi lấy nhiều mẫu SQL, chạy từng mẫu rồi **bỏ phiếu** theo đáp án. Khi các mẫu không đồng thuận, hệ thống **bỏ trống**
  thay vì đoán, vì câu trả lời sai bị trừ điểm.
- LoRA được train trên chính các lời giải đúng mà mô hình tìm ra trên tập train (rejection sampling).

| Phần | Nội dung | Phần cứng |
|---|---|---|
| 01 · EDA | dữ liệu gồm những gì: shop, tệp, kiểu đáp án, dạng câu hỏi, rulebook | CPU |
| 02 · Suy luận | chạy pipeline và ghi file nộp `submission.jsonl` | GPU bf16 từ 24 GB |
| 03 · Đánh giá | điểm PA, nhóm lỗi, ngưỡng bỏ phiếu | CPU |
| 04 · Dữ liệu train LoRA | dựng dữ liệu SFT từ các lời giải đúng | CPU, GPU cho bước ghi chú |
| 05 · Train LoRA | QLoRA trên Qwen3.5-9B | GPU bf16 từ 24 GB |

Chạy từ trên xuống. Mỗi phần có một ô tham số ở đầu phần. Khi không có GPU bf16 từ 24 GB (ví dụ GPU T4 trên Kaggle), các bước cần mô hình
dùng kết quả đã chạy sẵn trên GPU 24 GB, nên notebook vẫn chạy hết và vẫn vẽ được kết quả.

In [ ]:
# Setup. Local checkout: nothing to install. Kaggle: Internet on, GPU accelerator for inference / training, the competition data
# attached as a private dataset, and the code either attached as a dataset (a copy of the checkout) or cloned with your
# GitHub token in Add-ons -> Secrets as GITHUB_TOKEN (if the repo is private). Colab: Google Drive holding MyDrive/datamate/code and
# MyDrive/datamate/data (or a GITHUB_TOKEN secret for the code), a GPU runtime for inference / training.
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "github.com/hana200804/datamate-agent-Locked-In1.git"


def clone(root, token):
    subprocess.run(["git", "clone", "--depth", "1", f"https://x-access-token:{token}@{REPO_URL}", str(root)], check=True)
    subprocess.run(["git", "remote", "set-url", "origin", f"https://{REPO_URL}"], cwd=root, check=True)  # token out of saved files


if Path("/kaggle/input").exists():
    root = Path("/tmp/datamate-agent-Locked-In1")
    attached = sorted(p.parent.parent for p in Path("/kaggle/input").glob("**/datamate/notebook.py"))
    if not root.exists() and attached:
        shutil.copytree(attached[0], root)                         # code attached as a dataset: no token needed
    if not root.exists():
        from kaggle_secrets import UserSecretsClient
        clone(root, UserSecretsClient().get_secret("GITHUB_TOKEN"))
elif "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
    root, code = Path("/content/datamate-agent-Locked-In1"), Path("/content/drive/MyDrive/datamate/code")
    if not root.exists() and (code / "datamate/notebook.py").exists():
        shutil.copytree(code, root)                                # code uploaded to Drive: no token needed
    if not root.exists():
        from google.colab import userdata                          # key icon -> secret GITHUB_TOKEN with notebook access
        clone(root, userdata.get("GITHUB_TOKEN"))
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("base")            # each part below installs what it needs (Kaggle only)
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

## 01 · Khám phá dữ liệu (EDA)

Trước khi xây pipeline, ta xem dữ liệu trông như thế nào:
- mỗi cửa hàng (shop) có bao nhiêu câu hỏi và những tệp dữ liệu gì
- đáp án có những kiểu nào, câu hỏi thường hỏi gì
- rulebook quy định cách tính (doanh thu, hàng bán chạy, công nợ…) ra sao
- một shop sau khi nạp vào DuckDB trông thế nào: đây chính là phần schema mà mô hình nhìn thấy

In [ ]:
train_q, public_q = read(env.data / "train/questions.jsonl"), read(env.data / "public_test/questions.jsonl")
train_a = {a["id"]: a for a in read(env.data / "train/answers.jsonl")}
splits = {"train": train_q, "public": public_q}
def per_shop(qs):
    c = list(Counter(q["business_id"] for q in qs).values())
    return f"{min(c)} / {pd.Series(c).median():.0f} / {max(c)}"
pd.DataFrame({name: {"questions": len(qs), "shops": len({q["business_id"] for q in qs}), "questions per shop (min / median / max)": per_shop(qs),
                     "as_of_date range": f'{min(q["as_of_date"] for q in qs)} .. {max(q["as_of_date"] for q in qs)}'} for name, qs in splits.items()})

### Tệp dữ liệu của shop

Mỗi shop là một thư mục gồm README.md và các bảng (csv / xlsx / sqlite). Số bảng và dung lượng khác nhau rất nhiều.

In [ ]:
rows = []
for split, folder in (("train", "train"), ("public", "public_test")):
    for shop in sorted((env.data / folder / "data").iterdir()):
        files = [f for f in shop.rglob("*") if f.is_file() and f.name != "README.md"]
        rows.append({"split": split, "shop": shop.name, "files": len(files), "MB": round(sum(f.stat().st_size for f in files) / 2**20, 1),
                     **Counter(f.suffix.lower().lstrip(".") for f in files)})
files = pd.DataFrame(rows).fillna(0)
display(files.groupby("split")[[c for c in files.columns if c not in ("split", "shop")]].sum())
files.groupby("split")[["files", "MB"]].describe().round(1)

### Loại đáp án (train)

Phân bố type của đáp án chuẩn. `alias` là các cách viết khác được chấp nhận (mã hàng, cách viết khác trong dữ liệu).

In [ ]:
types = Counter(a["type"] for a in train_a.values())
pd.Series(types).sort_values().plot.barh(figsize=(6, 3), title="train answer types")
print({t: f"{n / len(train_a):.1%}" for t, n in types.most_common()})
print("answers with aliases:", sum(bool(a.get("alias")) for a in train_a.values()))

Mỗi kiểu đáp án một ví dụ:

In [ ]:
example = {}
for q in train_q:
    a = train_a[q["id"]]
    example.setdefault(a["type"], (q["business_id"], q["question"], a.get("answer")))
pd.DataFrame([{"type": t, "shop": shop, "question": text[:100], "answer": json.dumps(ans, ensure_ascii=False)[:70]}
              for t, (shop, text, ans) in example.items()])

### Dạng câu hỏi

Tỷ lệ câu hỏi chứa một số dạng khó, ví dụ khung thời gian ("tuần rồi", "quý 2"), công nợ, số tiền viết tắt ("3tr7") hay từ địa phương.
Pipeline tính sẵn gợi ý cho các dạng này (`datamate/hints.py`) để mô hình không phải tự suy đoán.

In [ ]:
import re
from datamate import hints as H
from datamate.ingest import norm
from datamate.prompts import SLANG, money_gloss
shapes = {
    "period named": lambda q: bool(H.period_hints(q["question"], q["as_of_date"])),
    "debt": lambda q: bool(H.DEBT_WORDS.search(norm(q["question"]))),
    "days by daily total": lambda q: bool(H.COUNT_DAYS.search(norm(q["question"])) and H.THRESHOLD.search(norm(q["question"]) + q["question"])),
    "profit": lambda q: bool(H.PROFIT.search(norm(q["question"]))),
    "delivery trips": lambda q: bool(H.TRIPS.search(norm(q["question"]))),
    "lots / expiry": lambda q: bool(H.LOT.search(norm(q["question"]))),
    "money shorthand": lambda q: bool(money_gloss(q["question"])),
    "chat words": lambda q: bool(set(re.findall(r"\w+", q["question"].lower())) & set(SLANG)),
}
share = pd.DataFrame({name: {k: sum(f(q) for q in qs) / len(qs) for k, f in shapes.items()} for name, qs in splits.items()})
share.plot.barh(figsize=(7, 4), title="share of questions")
share.style.format("{:.1%}")

In [ ]:
pd.DataFrame({name: pd.Series([len(q["question"]) for q in qs]).describe().round(0) for name, qs in splits.items()}).T

### Rulebook

Train và public có rulebook riêng. Public thêm một số luật (khách trung thành, giờ cao điểm, nhà cung cấp chính…).

In [ ]:
rb = {s: (env.data / f / "rulebook.md").read_text(encoding="utf-8") for s, f in (("train", "train"), ("public", "public_test"))}
rules = {s: {l.split("|")[1].strip() for l in t.splitlines() if l.startswith("| ") and not l.startswith("| Luật")} for s, t in rb.items()}
print(len(rules["train"]), "rules in train,", len(rules["public"]), "in public")
print("only in public:", sorted(rules["public"] - rules["train"]))
print("only in train:", sorted(rules["train"] - rules["public"]))

### Một shop sau khi nạp

`load_business` đọc mọi bảng vào DuckDB, `schema_text` là phần SCHEMA mà model nhìn thấy.

In [ ]:
from datamate.ingest import load_business, schema_text
db, meta = load_business(env.data / "train", "biz_101", canon=True)
display(pd.DataFrame([{"table": t["name"], "rows": t["rows"], "columns": len(t["columns"])} for t in meta["tables"]]))
print(schema_text(meta)[:3000])

## 02 · Suy luận (inference)

Pipeline xử lý từng shop qua năm bước:
1. **Nạp dữ liệu** vào DuckDB và dựng sẵn view chuẩn (`v_sales`, `v_orders`) cùng macro công nợ, để câu SQL ngắn và ít sai.
2. **Tạo prompt** gồm rulebook, schema, khung thời gian tính sẵn, vài ví dụ tương tự và câu hỏi.
3. **Lấy mẫu:** mô hình viết 8 câu SQL. Câu khó (các mẫu không đồng thuận) được thêm 4 mẫu ở chế độ thinking.
4. **Thực thi và sửa lỗi:** DuckDB chạy từng câu SQL. Câu bị lỗi được gửi lại cho mô hình kèm thông báo lỗi để sửa.
5. **Bỏ phiếu:** gom các đáp án giống nhau. Hệ thống chỉ trả lời khi nhóm lớn nhất đủ đồng thuận, ngược lại thì bỏ trống.

Kết quả là file `submission.jsonl` đúng định dạng nộp bài. `LIMIT = 10` chạy thử 10 câu, còn cả 1000 câu public mất khoảng 3,5 giờ
trên một GPU 24 GB.

In [ ]:
install("llm")

In [ ]:
SPLIT = "public_test"        # public_test | train | private_test (a folder of the data dir)
LIMIT = 10                   # first N questions (None = all)
IDS = None                   # a file of question ids, e.g. "splits/rft_train24.txt" or "splits/holdout600.txt"
VIEWS_FROM = None           # optional path to layers from your own earlier run
LORA = None                # base model; set a path to your own trained adapter to enable LoRA
SAVE_PROMPTS = True          # keep each question's prompt in the trace (needed to build SFT rows from a train run)
OUT_NAME = "nb_" + SPLIT

In [ ]:
out = env.runs / OUT_NAME
if env.can_run_llm:
    model = env.base_model()
    lora_flags = ["--lora", env.adapter(LORA), "--lora-thinking"] if LORA else []
    flags, engine = env.inference_flags()
    n = len(env.gpus)
    layers = ["--views-from", VIEWS_FROM] if VIEWS_FROM else []
    pick = (["--limit", str(LIMIT)] if LIMIT else []) + (["--ids", IDS] if IDS else []) + (["--save-prompts"] if SAVE_PROMPTS else [])
    make_cmd = lambda i: [env.python, "-m", "datamate.predict", "--data", env.data / SPLIT, "--output", out / f"shard{i}", "--shard", f"{i}/{n}",
                          "--model", model, *lora_flags, "--fewshot", "assets/fewshot_bank.jsonl", *flags, "--engine", engine, *layers, *pick]
    print(f"{n} GPU process(es) -> {out}\nflags: {' '.join(flags)}")
else:   # Kaggle T4 / P100 or a CPU session: the same run made on a 24 GB GPU (public_test, LIMIT = 10)
    env.use_saved(OUT_NAME, out)

In [ ]:
if env.can_run_llm:
    env.run_shards(make_cmd, n, out)

In [ ]:
from datamate.submission import check, merge_shards
merge_shards(out, env.data / SPLIT / "questions.jsonl")
errors = check(out / "submission.jsonl", env.data / SPLIT / "questions.jsonl")
print("format OK" if not errors else errors[:5])

### Kết quả

Số câu đã trả lời theo shop, vài câu mẫu cùng đáp án và tỷ lệ đồng thuận (`conf`). Với `SPLIT = "train"` có thêm điểm theo đáp án chuẩn.

In [ ]:
trace = [r for p in sorted(out.glob("shard*/trace.jsonl")) for r in read(p)]
df = pd.DataFrame([{"shop": r["business_id"], "answered": r["final"]["type"] != "abstain", "conf": r["conf"]} for r in trace])
display(df.groupby("shop").agg(questions=("answered", "size"), answered=("answered", "sum"), mean_conf=("conf", "mean")).round(2))
pd.DataFrame([{"question": r["question"][:90], "final": json.dumps(r["final"], ensure_ascii=False)[:80], "conf": round(r["conf"], 2)} for r in trace[:5]])

### Một câu hỏi đi qua pipeline

Câu hỏi có tỷ lệ đồng thuận thấp nhất (thường là câu khó, được thêm mẫu thinking): chế độ, câu SQL và đáp án của từng mẫu, rồi kết quả bỏ phiếu.
- Mẫu không viết ra được SQL hợp lệ (ví dụ ghi chú lặp mãi không dừng) không được tính phiếu.
- Tỷ lệ đồng thuận = số phiếu của đáp án thắng / tổng số mẫu.

In [ ]:
def last_sql(text):
    return " ".join(text.split("```sql")[-1].split("```")[0].split()) if "```sql" in text else ""

q = min(trace, key=lambda r: (r["conf"], -len(r["samples"])))
print(q["question"])
samples = pd.DataFrame([{"mẫu": k + 1,
                         "chế độ": "thinking" if "</think>" in s["text"] else "thường",
                         "đáp án": json.dumps(s["pred"]["answer"], ensure_ascii=False)[:50] if s.get("pred") else "",
                         "SQL": last_sql(s["text"])[:110] or "(lỗi: " + (s.get("error") or "").split(" | ")[0] + ")"}
                        for k, s in enumerate(q["samples"])])
display(samples)
votes = Counter(a for a in samples["đáp án"] if a)
print("bỏ phiếu:", dict(votes.most_common()), f"| {len(samples)} mẫu")
print("đáp án cuối:", json.dumps(q["final"], ensure_ascii=False), "| tỷ lệ đồng thuận:", round(q["conf"], 2))

In [ ]:
if (env.data / SPLIT / "answers.jsonl").exists():
    from datamate import scoring
    gold = {a["id"]: a for a in read(env.data / SPLIT / "answers.jsonl")}
    report, _ = scoring.evaluate({r["id"]: gold[r["id"]] for r in trace}, {r["id"]: r["final"] for r in trace},
                                 {q["id"]: q for q in read(env.data / SPLIT / "questions.jsonl")})
    print({k: report[k] for k in ("N", "correct", "wrong", "abstain", "PA", "precision")})

## 03 · Đánh giá

Điểm của cuộc thi là **PA (penalized accuracy)**: câu đúng +1, câu sai −1, câu bỏ trống 0, rồi chia cho số câu. Vì câu sai bị trừ điểm,
bỏ trống những câu không chắc thường có lợi hơn là đoán.

Phần này gồm:
1. Chấm một lượt chạy trên **train** (có đáp án): điểm tổng, theo shop, theo dạng câu, nhóm lỗi và ngưỡng bỏ phiếu.
2. So các **bản nộp public** với nhau, không cần nhãn.

In [ ]:
RUN = "runs/nb_train"     # a train run (its trace.jsonl, or shard*/trace.jsonl), e.g. runs/nb_train from 02_inference
SUBS = {"base": "runs/public_base", "lora": "runs/public_lora"}   # public submissions to compare

In [ ]:
from datamate import scoring, voting
from datamate.hints import DEBT_WORDS
run = env.root / RUN if not str(RUN).startswith("/") else Path(RUN)
gold = {a["id"]: a for a in read(env.data / "train/answers.jsonl")}
trace = {r["id"]: r for p in ([run / "trace.jsonl"] if (run / "trace.jsonl").exists() else sorted(run.glob("shard*/trace.jsonl"))) for r in read(p)}
print(len(trace), "traced questions")

### 1. Điểm của lượt chạy train

- **PA:** điểm của cuộc thi.
- **precision:** tỷ lệ đúng trong các câu có trả lời. **coverage:** tỷ lệ câu có trả lời.
- **Khoảng tin cậy:** lấy mẫu lại theo shop (bootstrap), cho biết điểm dao động bao nhiêu nếu đổi sang tập shop khác.

In [ ]:
questions = {q["id"]: q for q in read(env.data / "train/questions.jsonl")}
report, rows = scoring.evaluate({i: gold[i] for i in trace}, {i: r["final"] for i, r in trace.items()}, questions)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in report.items() if k in ("N", "correct", "wrong", "abstain", "PA", "precision", "coverage")})
print("PA 95% CI (shop bootstrap):", [round(x, 3) for x in report["PA_cluster_bootstrap_95"]])
pd.DataFrame(report["breakdowns"]["gold_type"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("N", ascending=False)

In [ ]:
by_shop = pd.DataFrame(report["breakdowns"]["business_id"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("PA")
by_shop.plot.barh(y="PA", figsize=(6, 7), legend=False, title="PA by shop")
by_shop.head(8)

### Nhóm lỗi

Mỗi câu chưa đúng thuộc một nhóm: không mẫu nào đúng (cần model / prompt tốt hơn), có mẫu đúng nhưng nhóm sai thắng phiếu, hoặc nhóm đúng
đứng đầu nhưng dưới ngưỡng nên bỏ trống (cần bỏ phiếu / ngưỡng tốt hơn).

In [ ]:
def kind(r):
    g = gold[r["id"]]
    s = scoring.score(r["final"], g)
    if s == 1: return "right"
    if not any(p and scoring.score(p, g) == 1 for p in (x["pred"] for x in r["samples"])): return "no right sample"
    if s == -1: return "wrong: a wrong group won"
    top_right = bool(r.get("vote")) and scoring.score(r["vote"], g) == 1
    return "abstain: right group on top, below threshold" if top_right else "abstain: right sample, another group on top"
pd.Series(Counter(kind(r) for r in trace.values())).sort_values(ascending=False).to_frame("questions")

### Ngưỡng bỏ phiếu và oracle@K

- **Ngưỡng bỏ phiếu:** tỷ lệ mẫu đồng thuận tối thiểu để trả lời. Ngưỡng cao thì ít sai hơn nhưng bỏ trống nhiều hơn. Biểu đồ bỏ phiếu lại
  trên các mẫu đã lưu với từng ngưỡng.
- **oracle@K:** tỷ lệ câu có ít nhất một mẫu đúng trong K mẫu, tức mức điểm tối đa nếu luôn chọn được mẫu đúng.

In [ ]:
def replay(r, tau):
    texts, preds = [s["text"] for s in r["samples"]], [s["pred"] for s in r["samples"]]
    pool = voting.debt_pool(texts, 8, bool(DEBT_WORDS.search(norm(r["question"]))))
    ans, conf = voting.vote([preds[j] for j in pool], len(pool))
    return voting.final_answer(ans, conf, tau, r["final"].get("reason") == "period outside the shop's data dates", True)
sweep = pd.Series({t: sum(scoring.score(replay(r, t), gold[i]) for i, r in trace.items()) / len(trace) for t in (0.25, 0.375, 0.5, 0.625, 0.75)}, name="PA")
oracle = sum(any(p and scoring.score(p, gold[i]) == 1 for p in (s["pred"] for s in r["samples"])) for i, r in trace.items()) / len(trace)
print(f"oracle (any sample right): {oracle:.3f}")
sweep.to_frame()

### Theo dạng câu hỏi

In [ ]:
shape = lambda q: ("debt" if DEBT_WORDS.search(norm(q)) else "profit" if H.PROFIT.search(norm(q)) else "trips" if H.TRIPS.search(norm(q))
                   else "lots" if H.LOT.search(norm(q)) else "other")
df = pd.DataFrame([{"shape": shape(r["question"]), "score": scoring.score(r["final"], gold[i])} for i, r in trace.items()])
df.groupby("shape")["score"].agg(n="size", PA="mean").sort_values("n", ascending=False).round(3)

### 2. So các bản nộp public (không nhãn)

Số câu trả lời, mức trùng đáp án giữa các file, và chỗ khác nhau theo shop giữa hai bản cuối.

In [ ]:
def key(a):
    t, x = a.get("type"), a.get("answer")
    if t == "abstain" or (x is None and t != "unanswerable"): return "ABSTAIN"
    return "UNANSWERABLE" if t == "unanswerable" else json.dumps([t, round(x, 2) if isinstance(x, float) else x], ensure_ascii=False, sort_keys=True)
pq = {q["id"]: q for q in read(env.data / "public_test/questions.jsonl")}
subs = {name: {r["id"]: key(r) for r in read(env.root / p / "submission.jsonl")} for name, p in SUBS.items() if (env.root / p / "submission.jsonl").exists()}
pd.DataFrame({n: {"answered": sum(v != "ABSTAIN" for v in s.values()), "unanswerable": sum(v == "UNANSWERABLE" for v in s.values())} for n, s in subs.items()}).T

In [ ]:
names = list(subs)
pd.DataFrame([[sum(subs[a][i] == subs[b][i] for i in pq) / len(pq) for b in names] for a in names], index=names, columns=names).style.format("{:.1%}")

In [ ]:
if len(names) >= 2:
    a, b = names[-2], names[-1]
    display(pd.DataFrame([{"shop": q["business_id"], "changed": subs[a][i] != subs[b][i], "newly answered": subs[a][i] == "ABSTAIN" != subs[b][i],
                           "newly abstained": subs[b][i] == "ABSTAIN" != subs[a][i]} for i, q in pq.items()]).groupby("shop").sum())

## 04 · Dữ liệu train LoRA

LoRA học từ chính các lời giải đúng mà mô hình tìm ra trên tập train (rejection sampling):
1. **Chạy pipeline trên train** (có đáp án). Mỗi câu có 8 mẫu SQL, câu khó có thêm mẫu thinking.
2. **Giữ lời giải đúng:** so đáp án của từng mẫu với đáp án chuẩn. Ưu tiên lời giải dùng macro công nợ, rồi view chuẩn, cuối cùng mới tới
   bảng thô. Câu khó (ít mẫu đúng) được giữ nhiều dòng hơn câu dễ.
3. **Ghi chú trước SQL:** thêm vài dòng "Notes:" (khung thời gian, cách tính) trước câu SQL, đúng định dạng mô hình viết lúc suy luận.
   Bước này cần GPU.

Chỉ dùng dữ liệu train. Câu public / private không bao giờ được đưa vào dữ liệu train.

In [ ]:
install("llm")

In [ ]:
RUNS = ["runs/sql_v8_pool"]        # labeled train runs with --save-prompts (dirs holding trace.jsonl or shard*/trace.jsonl)
TEACHER_RUNS = []                  # optional teacher runs on train questions
OUT = "runs/sft/sql_rft_nb.jsonl"
EASY = 800                         # rows from easy questions (6-8 of 8 samples right)
MAX = 4000                         # cap on rows (debt and teacher rows are kept)
ADD_NOTES = True                   # needs a GPU (vLLM, base weights)
BUILD_BANK = False                 # also write a few-shot bank from RUNS

In [ ]:
def trace_dir(r):  # build_sft.py reads RUN/trace.jsonl: join the shards of a multi-GPU run first
    r = Path(r) if Path(r).is_absolute() else env.root / r
    if not (r / "trace.jsonl").exists() and list(r.glob("shard*/trace.jsonl")):
        with (r / "trace.jsonl").open("w", encoding="utf-8") as f:
            for p in sorted(r.glob("shard*/trace.jsonl")):
                f.write(p.read_text(encoding="utf-8"))
    return r
runs, teachers = [trace_dir(r) for r in RUNS], [trace_dir(r) for r in TEACHER_RUNS]
out = Path(OUT) if Path(OUT).is_absolute() else env.root / OUT
out.parent.mkdir(parents=True, exist_ok=True)
env.run([env.python, "training/build_sft.py", *runs, *(["--teacher", *teachers] if teachers else []), "--fewshot", "assets/fewshot_bank.jsonl",
         "--out", out, "--max", MAX, "--easy", EASY], log=out.with_suffix(".log"), show="available|rows by|Traceback|Error")

In [ ]:
sft = out
if ADD_NOTES and not env.can_run_llm:                   # the notes step needs the model: take the notes file saved from a 24 GB GPU run
    sft = env.use_saved(out.stem + "_notes.jsonl", out.with_name(out.stem + "_notes.jsonl"))
elif ADD_NOTES:
    flags, _ = env.inference_flags()                     # checks the GPU, Ampere runs bf16 (no fp8)
    sft = out.with_name(out.stem + "_notes.jsonl")
    env.run([env.python, "training/add_notes.py", out, sft, "--model", env.base_model(), "--rulebook", env.data / "train/rulebook.md",
             "--quant", "fp8" if "--quant" in flags else "none"], log=sft.with_suffix(".log"), show="notes on|Traceback|Error")
print("SFT file for 05_train_lora:", sft)

### Dữ liệu SFT trông thế nào

Số dòng theo route, tỷ lệ dòng thinking, độ dài (prompt chiếm phần lớn mỗi dòng, loss chỉ tính trên phần trả lời).

In [ ]:
rows = read(sft)
df = pd.DataFrame([{"id": r["id"], "route": r.get("route"), "thinking": bool(r.get("thinking")), "prompt_chars": len(r["system"]) + len(r["user"]),
                    "reply_chars": len(r["reply"]), "notes": r["reply"].startswith("Notes:")} for r in rows])
print(len(df), "rows,", df["id"].nunique(), "questions")
display(df.groupby(["route", "thinking"]).size().unstack(fill_value=0))
display(df[["prompt_chars", "reply_chars"]].describe().round(0))
print(f"plain rows with notes: {df[~df.thinking].notes.mean():.1%}")

In [ ]:
r = rows[0]
print(r["user"][-600:], "\n---- reply ----\n", r["reply"][:800])

In [ ]:
if BUILD_BANK:
    env.run([env.python, "training/build_fewshot_bank.py", *runs, "--out", out.with_name("fewshot_bank.jsonl")], show="Traceback|Error|bank|rows")

## 05 · Train LoRA

- **QLoRA:** mô hình gốc Qwen3.5-9B được nạp ở dạng 4-bit và giữ nguyên, chỉ train các ma trận LoRA hạng 16 gắn vào attention và MLP
  (chưa tới 1% số tham số), nên vừa một GPU 24 GB.
- **Loss chỉ tính trên phần trả lời:** mỗi dòng dài 9k đến 19k token vì prompt chứa rulebook và schema, nhưng loss chỉ tính trên phần
  ghi chú và SQL mà mô hình cần viết (khoảng 4% số token).
- **Unsloth** giúp train nhanh và tiết kiệm bộ nhớ. Khởi tạo adapter mới từ model gốc; thời gian train phải đo trên lượt chạy riêng.

`LIMIT_ROWS` và `MAX_STEPS` cho phép chạy thử vài bước để xem quy trình và đường loss.

In [ ]:
install("train")  # Kaggle: a separate session for this part (the training stack differs from vLLM)

In [ ]:
SFT = "runs/sft/sql_rft_nb_notes.jsonl"   # from 04_training_data
INIT_ADAPTER = None                     # train a new adapter from the base model
OUT = "runs/lora/sql_rft_nb"
MAX_LEN = 32768                            # tokens per row, longer rows are dropped
LR = 1e-4                                  # fresh adapter
EPOCHS = 1
GRAD_ACCUM = 8
LIMIT_ROWS = None                          # e.g. 64 for a short learning run
MAX_STEPS = -1                             # e.g. 3 for a speed / memory check
SAVE_STEPS = 50
RESUME = True                              # continue from the newest checkpoint in OUT
COMPLETION_LOGITS = True                   # LM head only on reply positions: same loss, ~12% faster

In [ ]:
py = str(env.root / ".venv-train/bin/python") if (env.root / ".venv-train/bin/python").exists() else env.python   # the training stack
out = Path(OUT) if Path(OUT).is_absolute() else env.root / OUT
sft = Path(SFT) if Path(SFT).is_absolute() else env.root / SFT
assert sft.exists(), f"{sft} not found: build it in 04_training_data"
print("training python:", py, "| precision:", env.training_precision(), "| rows:", sum(1 for _ in open(sft, encoding="utf-8")))

### Kernel fla

24 trên 32 lớp của Qwen3.5-9B là Gated DeltaNet, một dạng attention tuyến tính. Thư viện `fla-core` cung cấp kernel Triton nhanh hơn
khoảng 90 lần so với bản PyTorch tham chiếu. Ô này kiểm tra kernel cho kết quả khớp với bản tham chiếu trước khi train.

In [ ]:
extra_env = {}
if not env.can_run_llm:
    print("no bf16 GPU with >= 24 GB: kernel check skipped")
else:
    try:
        env.run([py, "training/check_fla.py", "--T", "2048"], show="fla check|T=|rel L2|Traceback|Error")
    except RuntimeError as e:
        print("fla unusable here, training without it:", e)
        extra_env = {"PYTHONPATH": str(env.root / "training/no_fla")}

In [ ]:
if env.can_run_llm:
    cmd = [py, "training/train_lora.py", sft, "--out", out, "--model", env.base_model(), "--max-len", MAX_LEN, "--r", 16, "--alpha", 32, "--lr", LR,
           "--epochs", EPOCHS, "--grad-accum", GRAD_ACCUM, "--end", "<|im_end|>", "--no-thinking", "--max-steps", MAX_STEPS, "--save-steps", SAVE_STEPS]
    cmd += (["--init-adapter", env.adapter(INIT_ADAPTER)] if INIT_ADAPTER else []) + (["--limit-rows", LIMIT_ROWS] if LIMIT_ROWS else [])
    cmd += (["--resume"] if RESUME else []) + (["--completion-logits"] if COMPLETION_LOGITS else [])
    env.run(cmd, log=out.with_name(out.name + ".log"), env=extra_env, show="examples|tokens:|precision|resuming|continuing|'loss'|steps in|Traceback|Error|OutOfMemory", every=120)
else:   # Kaggle T4 / P100 or a CPU session: the log of the same run made on a 24 GB GPU
    env.use_saved(out.name + ".log", out.with_name(out.name + ".log"))

### Loss

Loss theo bước (đọc từ `trainer_state.json` của checkpoint mới nhất, hoặc từ log).

In [ ]:
import ast
log_lines = out.with_name(out.name + ".log").read_text(encoding="utf-8", errors="replace").replace("\r", "\n").splitlines()
losses = [float(ast.literal_eval(l.strip())["loss"]) for l in log_lines if l.strip().startswith("{'loss'")]
if losses:
    s = pd.Series(losses)
    ax = s.plot(alpha=0.3, figsize=(7, 3), title="training loss")
    s.rolling(max(1, len(s) // 20)).mean().plot(ax=ax)
print(len(losses), "logged steps; adapter:", out / "adapter", "exists" if (out / "adapter/adapter_model.safetensors").exists() else "not here (saved log only)" if not env.can_run_llm else "MISSING")

### Dùng adapter vừa train

Đặt `LORA` ở phần 02 bằng đường dẫn `OUT/adapter` (ví dụ `"runs/lora/sql_rft_nb/adapter"`) để chạy suy luận với adapter vừa train,
rồi so điểm với model gốc ở phần 03.